# Montpelier Evacuation Route Planner
## A* Shortest Path Algorithm for Emergency Evacuation

This notebook demonstrates the A* pathfinding algorithm for finding optimal evacuation routes in Montpelier, Vermont.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import folium

from data_loader import load_or_download_network, get_node_nearest_to_point
from routing import a_star_path, dijkstra_path, compare_algorithms, RouteResult
from visualization import create_route_map, create_comparison_map, create_static_route_map
from output import export_all_formats

## 2. Load Road Network

Load the road network for Montpelier, Vermont. The network is cached locally.

In [ ]:
# Load the road network (10km radius from city center)
G = load_or_download_network(dist_km=10, cache_path='../data/montpelier_network.pkl')

print(f"\nNetwork Statistics:")
print(f"  Nodes: {len(G.nodes)}")
print(f"  Edges: {len(G.edges)}")
print(f"  Average node degree: {sum(dict(G.degree()).values()) / len(G.nodes):.2f}")

## 3. Define Source and Destination

We'll find a route from downtown Montpelier to the southern edge of our coverage area.

In [ ]:
# Define source (downtown Montpelier) and destination (south of city)
source_point = (44.2601, -72.5754)  # Montpelier downtown
dest_point = (44.1800, -72.5700)    # South of Montpelier

# Find nearest network nodes
source_node = get_node_nearest_to_point(G, source_point)
dest_node = get_node_nearest_to_point(G, dest_point)

print(f"Source: {source_point} -> Node {source_node}")
print(f"  Node coordinates: ({G.nodes[source_node]['y']:.6f}, {G.nodes[source_node]['x']:.6f})")
print(f"\nDestination: {dest_point} -> Node {dest_node}")
print(f"  Node coordinates: ({G.nodes[dest_node]['y']:.6f}, {G.nodes[dest_node]['x']:.6f})")

## 4. Run A* Algorithm

The A* algorithm uses travel time as the edge cost and straight-line distance as the heuristic.

In [ ]:
# Run A* algorithm
start_time = time.time()
astar_result = a_star_path(G, source_node, dest_node)
astar_time = time.time() - start_time

if astar_result:
    print("A* Route Found!")
    print(f"\nRoute Statistics:")
    print(f"  Total Distance: {astar_result.distance_km:.2f} km")
    print(f"  Total Travel Time: {astar_result.travel_time_minutes:.1f} minutes")
    print(f"  Number of Nodes: {len(astar_result.path)}")
    print(f"  Number of Segments: {len(astar_result.edges)}")
    print(f"  Computation Time: {astar_time*1000:.2f} ms")
else:
    print("No path found!")

## 5. Compare with Dijkstra Baseline

Compare A* with Dijkstra's algorithm to validate our implementation.

In [ ]:
# Compare algorithms
astar_result, dijkstra_result, metrics = compare_algorithms(G, source_node, dest_node)

print("Algorithm Comparison:")
print("=" * 50)
print(f"\n{'Metric':<30} {'A*':>10} {'Dijkstra':>10}")
print("-" * 50)
print(f"{'Distance (km)':<30} {astar_result.distance_km:>10.2f} {dijkstra_result.distance_km:>10.2f}")
print(f"{'Travel Time (min)':<30} {astar_result.travel_time_minutes:>10.1f} {dijkstra_result.travel_time_minutes:>10.1f}")
print(f"{'Nodes':<30} {len(astar_result.path):>10} {len(dijkstra_result.path):>10}")
print(f"{'Computation Time (ms)':<30} {metrics['astar_time_sec']*1000:>10.2f} {metrics['dijkstra_time_sec']*1000:>10.2f}")
print("-" * 50)
print(f"{'Paths Identical':<30} {'Yes' if metrics['paths_equal'] else 'No':>10}")

## 6. Visualize the Route

In [ ]:
# Create interactive map
m = create_route_map(
    G, astar_result,
    output_path='../outputs/evacuation_route_map.html',
    show_network=True,
    start_point=source_point,
    end_point=dest_point
)

# Display the map
m

In [ ]:
# Create comparison map
comparison_map = create_comparison_map(
    G, astar_result, dijkstra_result,
    labels=('A* Route', 'Dijkstra Route'),
    output_path='../outputs/route_comparison_map.html'
)

# Display the comparison map
comparison_map

In [ ]:
# Create static matplotlib map
fig = create_static_route_map(
    G, astar_result,
    output_path='../outputs/evacuation_route_static.png'
)
plt.show()

## 7. Export Route Data

Export the route to various formats for use in other applications.

In [ ]:
# Export route in all formats
outputs = export_all_formats(astar_result, G, output_dir='../outputs')

print("\nExported Files:")
for fmt, path in outputs.items():
    print(f"  {fmt}: {path}")

## 8. Analyze Route Segments

Break down the route by road type and analyze segment properties.

In [ ]:
# Create a DataFrame of route segments
segment_data = []
for i, (u, v, data) in enumerate(astar_result.edges):
    segment_data.append({
        'segment_id': i + 1,
        'from_node': u,
        'to_node': v,
        'length_m': data.get('length', 0),
        'travel_time_sec': data.get('travel_time', 0),
        'speed_kmh': data.get('speed_kmh', 0),
        'highway_type': data.get('highway', 'unknown'),
        'road_name': data.get('name', 'unnamed')
    })

segments_df = pd.DataFrame(segment_data)
segments_df.head(10)

In [ ]:
# Summarize by highway type
highway_summary = segments_df.groupby('highway_type').agg({
    'segment_id': 'count',
    'length_m': 'sum',
    'travel_time_sec': 'sum'
}).rename(columns={
    'segment_id': 'segment_count',
    'length_m': 'total_length_m',
    'travel_time_sec': 'total_time_sec'
})

highway_summary['total_length_km'] = highway_summary['total_length_m'] / 1000
highway_summary['total_time_min'] = highway_summary['total_time_sec'] / 60

highway_summary.sort_values('total_length_km', ascending=False)

In [ ]:
# Visualize segment lengths
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Length distribution
axes[0].hist(segments_df['length_m'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Segment Length (m)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Segment Lengths')

# Travel time distribution
axes[1].hist(segments_df['travel_time_sec'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Travel Time (seconds)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Segment Travel Times')

plt.tight_layout()
plt.savefig('../outputs/segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Multiple Route Analysis

Find routes from the same source to multiple potential destinations.

In [ ]:
# Define multiple destinations (safety shelters outside the city)
destinations = [
    (44.1800, -72.5700),  # South
    (44.3200, -72.5800),  # North
    (44.2500, -72.7000),  # West
    (44.2600, -72.4500),  # East
]

dest_labels = ['South', 'North', 'West', 'East']

# Find routes to each destination
routes = []
for i, dest in enumerate(destinations):
    dest_node = get_node_nearest_to_point(G, dest)
    route = a_star_path(G, source_node, dest_node)
    if route:
        routes.append({
            'destination': dest_labels[i],
            'distance_km': route.distance_km,
            'travel_time_min': route.travel_time_minutes,
            'nodes': len(route.path),
            'route': route
        })

# Display comparison
routes_df = pd.DataFrame(routes)
print("Route Comparison from Downtown Montpelier:")
print("=" * 60)
routes_df[['destination', 'distance_km', 'travel_time_min', 'nodes']]

In [ ]:
# Create multi-route map
from visualization import create_multi_route_map

route_list = [r['route'] for r in routes]
m = create_multi_route_map(G, route_list, output_path='../outputs/multi_route_map.html')

# Display map
m

## 10. Summary

This notebook demonstrated:

1. **Road Network Loading**: Using osmnx to download and cache the Montpelier road network

2. **A* Pathfinding**: Finding optimal routes using travel time as edge cost and straight-line distance as heuristic

3. **Algorithm Comparison**: Validating A* results against Dijkstra baseline

4. **Visualization**: Creating interactive (Folium) and static (Matplotlib) route maps

5. **Data Export**: Exporting routes to CSV, GeoJSON, and text summary formats

6. **Route Analysis**: Breaking down routes by road type and segment properties